# BERT-100M data-movement (I/O) study — weight-stationary crossbar mapping

Each trained matmul is mapped **weight-stationary** onto a fixed 2D CiM crossbar fabric
sized exactly to it (`system=fetch_weights_hbm2_ws`, grid = `round_up_even(M/32) ×
round_up_even(C/32)`): every weight is written **once** (0 reloads) and every crossbar is
used. Reports **chip-level** I/O (weight/activation dims, tiles used, weight/activation/
output data moved + iterations) and **crossbar-level** I/O (data in/out per crossbar),
then validates the equations against the simulator's measured NoC volumes.

### Equations (weight-stationary)

Per crossbar (`array_row × array_col`, here 32×32):
```
weight_in  = array_row × array_col × weight_bits / 8      # programmed ONCE
data_in    = array_row × input_bits  / 8                  # per cycle
data_out   = array_col × output_bits / 8                  # per cycle
```
Chip level:
```
tiles_parallel     = (M/array_col) × (C/array_row) × G           # ALL blocks resident
total_weight_moved = M × C × G × weight_bits / 8                 # into chip, moved ONCE
total_input_moved  = C × P × G × input_bits  / 8   over P iters  # input streamed once
output_size        = M × P × G × output_bits / 8                # final output tensor
output_moved       = output_size × (C / array_row)              # ALL partial outputs leaving
                   = tiles_parallel × (array_col × out_bits/8) × P  # crossbars (accumulation)
```
`tiles_parallel` = active crossbars used; `G` = independent matmuls in parallel (head count);
`P` = tokens streamed. **Output note:** each of the M×P outputs is a sum of `C/array_row`
partial products, so the real output *movement* is `output_size × C/array_row`, not just the
size. The simulator's `col_noc` volume alone under-counts this on tall grids because the
mapper splits the C-reduction across levels (some within a column on `col_noc`, the rest
across columns above it) — see the validation cell.

In [1]:
from _import_scripts import scripts
from scripts.utils import get_spec, run_mapper_with_crossbar_log
import pandas as pd


In [2]:
HIDDEN_SIZE = 768
N_HEADS = 12
HEAD_DIM = HIDDEN_SIZE // N_HEADS  # 64
INTERMEDIATE_SIZE = 3072
N_LAYERS = 12
BATCH = 1
SEQ_LEN = 128

MACRO = 'basic_analog'
CHIP = 'cim_tile_array'
SYSTEM = 'fetch_weights_hbm2_ws'   # WEIGHT-STATIONARY: main_memory forbids weight reloads
XB = 32                            # crossbar logical size: 32 out-ch (M) x 32 in-ch (C)

def round_up_even(n):
    n = int(n)
    return n if n % 2 == 0 else n + 1

# Per-matmul (M, C, P, G). qkv/attn_out/ffn1/ffn2 are trained (weight-stationary);
# qk/av are dynamic attention (G = head count).
SHAPES = {
    'qkv_proj':      dict(M=3 * HIDDEN_SIZE, C=HIDDEN_SIZE, P=SEQ_LEN, G=1),
    'qk':            dict(M=SEQ_LEN, C=HEAD_DIM, P=SEQ_LEN, G=N_HEADS),
    'av':            dict(M=HEAD_DIM, C=SEQ_LEN, P=SEQ_LEN, G=N_HEADS),
    'attn_out_proj': dict(M=HIDDEN_SIZE, C=HIDDEN_SIZE, P=SEQ_LEN, G=1),
    'ffn1':          dict(M=INTERMEDIATE_SIZE, C=HIDDEN_SIZE, P=SEQ_LEN, G=1),
    'ffn2':          dict(M=HIDDEN_SIZE, C=INTERMEDIATE_SIZE, P=SEQ_LEN, G=1),
}

# Weight-stationary grid PER matmul: cols hold M-blocks (outputs), rows hold C-blocks
# (inputs), sized exactly to the layer (round up to even) so weights are written once.
GRID = {name: (round_up_even(cfg['M'] / XB), round_up_even(cfg['C'] / XB))  # (cols, rows)
        for name, cfg in SHAPES.items()}


## Build and run each layer shape — weight-stationary

Each matmul is mapped onto its **own fixed crossbar region**, sized exactly to the layer:
`tile_cols = round_up_even(M/32)`, `tile_rows = round_up_even(C/32)` (see `GRID`). With
`system=fetch_weights_hbm2_ws` the mapper is forbidden from re-fetching weights in time
(`main_memory: no_iteration_over_dataspaces: [Weights]`), so every weight is written once
and every tile is used — 0 reloads, 100% utilization. `dnn='msft_phi_1_5', layer='001'`
is histogram scaffolding only; the real geometry is set via `spec.problem.instance`.

In [3]:
def build_spec(shape_name: str):
    cfg = SHAPES[shape_name]
    tile_cols, tile_rows = GRID[shape_name]          # (cols=M-blocks, rows=C-blocks)
    spec = get_spec(
        macro=MACRO, chip=CHIP, system=SYSTEM,
        dnn='msft_phi_1_5', layer='001',  # histogram scaffolding only
        tile_rows=tile_rows, tile_cols=tile_cols,
    )
    spec.problem.instance.update({
        'M': cfg['M'], 'C': cfg['C'], 'P': cfg['P'], 'G': cfg['G'], 'N': BATCH,
    })
    return spec


def run_shape(shape_name: str):
    spec = build_spec(shape_name)
    # stats = Accelergy energy/area/cycles; crossbar_df = per-crossbar placement;
    # edges_df = simulator-measured inter-crossbar NoC byte volumes (for validation).
    stats, crossbar_df, edges_df = run_mapper_with_crossbar_log(spec)
    return shape_name, stats, crossbar_df, edges_df


results = {name: run_shape(name) for name in SHAPES}


In [4]:
def crossbars_used(cdf):
    # active crossbars = unique tile coordinates (all '*_idx' cols except the
    # main_memory time-step index); robust to one- or two-dim tile levels.
    idc = [c for c in cdf.columns if c.endswith('_idx') and 'main_memory' not in c]
    return len(cdf[idc].drop_duplicates()) if idc else 1


# ================= CHIP-LEVEL I/O (weight-stationary) =================
chip = []
for name, cfg in SHAPES.items():
    st, cdf = results[name][1], results[name][2]
    M, C, P, G = cfg['M'], cfg['C'], cfg['P'], cfg['G']
    wbits, ibits, obits = st.weight_bits, st.input_bits, st.output_bits
    acol = int(cdf['array_cols'].iloc[0])
    tiles = crossbars_used(cdf)
    chip.append({
        'layer':               name,
        'weight_dim(MxC)':     f'{M}x{C}',
        'act_in_dim(CxP)':     f'{C}x{P}',
        'out_dim(MxP)':        f'{M}x{P}',
        'G':                   G,
        'tiles_parallel':      tiles,                              # active crossbars used
        'total_weight_moved_B': M * C * G * wbits // 8,            # into chip, ONCE
        'activation_moved_B':  C * P * G * ibits // 8,             # input streamed
        'act_iterations':      P,                                  # tokens to process activation
        'output_size_B':       M * P * G * obits // 8,             # final output tensor
        'output_moved_B':      tiles * acol * obits // 8 * P,      # all partial outputs off crossbars
        'out_iterations':      P,                                  # tokens to produce output
    })
chip_io = pd.DataFrame(chip).set_index('layer')

print("CHIP-LEVEL I/O  (weight-stationary, write-once — 0 weight reloads)")
print(chip_io.to_string())


CHIP-LEVEL I/O  (weight-stationary, write-once — 0 weight reloads)
              weight_dim(MxC) act_in_dim(CxP) out_dim(MxP)   G  tiles_parallel  total_weight_moved_B  activation_moved_B  act_iterations  output_size_B  output_moved_B  out_iterations
layer                                                                                                                                                                                  
qkv_proj             2304x768         768x128     2304x128   1            1728               1769472               98304             128         294912         7077888             128
qk                     128x64          64x128      128x128  12              96                 98304               98304             128         196608          393216             128
av                     64x128         128x128       64x128  12              96                 98304              196608             128          98304          393216             128
attn_out_proj

In [5]:
# ================= CROSSBAR-LEVEL I/O (per single crossbar) =================
# Per cycle:  data_in = array_row x input_bits/8 ,  data_out = array_col x output_bits/8.
# Weight is written ONCE (array_row x array_col x weight_bits/8). Over the full inference
# each crossbar sees P cycles (P tokens streamed through its stationary weights).
xb = []
for name, cfg in SHAPES.items():
    st, cdf = results[name][1], results[name][2]
    P = cfg['P']
    arow, acol = int(cdf['array_rows'].iloc[0]), int(cdf['array_cols'].iloc[0])
    wbits, ibits, obits = st.weight_bits, st.input_bits, st.output_bits
    din, dout = arow * ibits // 8, acol * obits // 8
    xb.append({
        'layer':               name,
        'array(row x col)':    f'{arow}x{acol}',
        'weight_in_B(once)':   arow * acol * wbits // 8,
        'data_in_per_cycle_B': din,
        'data_out_per_cycle_B': dout,
        'data_in_total_B':     din * P,     # over P tokens
        'data_out_total_B':    dout * P,
    })
xbar_io = pd.DataFrame(xb).set_index('layer')

print("CROSSBAR-LEVEL I/O  (one crossbar; totals over P tokens)")
print(xbar_io.to_string())


CROSSBAR-LEVEL I/O  (one crossbar; totals over P tokens)
              array(row x col)  weight_in_B(once)  data_in_per_cycle_B  data_out_per_cycle_B  data_in_total_B  data_out_total_B
layer                                                                                                                          
qkv_proj                 32x32               1024                   32                    32             4096              4096
qk                       32x32               1024                   32                    32             4096              4096
av                       32x32               1024                   32                    32             4096              4096
attn_out_proj            32x32               1024                   32                    32             4096              4096
ffn1                     32x32               1024                   32                    32             4096              4096
ffn2                     32x32               10

In [6]:
# ================= VALIDATION: equations vs. simulator-measured I/O =================
# Compare the closed-form equations against the simulator's ACTUAL measured NoC byte
# volumes (Timeloop .stats.txt, parsed by crossbar_stats: row_noc = input broadcast,
# col_noc = output accumulation).
def noc_volume(edf, comp, ds):
    s = edf[(edf['component'] == comp) & (edf['dataspace'] == ds)]['total_volume_bytes']
    return int(s.sum())

val = []
for name, cfg in SHAPES.items():
    st, cdf, edf = results[name][1], results[name][2], results[name][3]
    M, C, P, G = cfg['M'], cfg['C'], cfg['P'], cfg['G']
    ibits, obits = st.input_bits, st.output_bits
    acol = int(cdf['array_cols'].iloc[0])
    tiles = crossbars_used(cdf)
    eq_in      = C * P * G * ibits // 8                  # input activation tensor
    out_moved  = tiles * acol * obits // 8 * P           # true output movement (sum over crossbars)
    sim_in     = noc_volume(edf, 'row_noc', 'Inputs')    # measured input broadcast
    sim_colnoc = noc_volume(edf, 'col_noc', 'Outputs')   # measured col_noc slice (within-column only)
    val.append({
        'layer': name,
        'eq_input_B': eq_in, 'sim_input_B': sim_in,
        'in_ratio': round(sim_in / eq_in, 3) if eq_in else None,
        'output_moved_B': out_moved,                     # = output_size x (C/array_row)
        'sim_col_noc_B': sim_colnoc,                     # simulator's col_noc (partial slice)
        'col_noc/moved': round(sim_colnoc / out_moved, 2) if out_moved else None,
    })
val_df = pd.DataFrame(val).set_index('layer')

print("VALIDATION: equations vs simulator-measured NoC volumes")
print(val_df.to_string())
print("\nInput  : sim/eq = 1.0  ->  input equation VALIDATED against the simulator.")
print("Output : output_moved = tiles x (array_col x out_bits/8) x P = output_size x C/array_row")
print("         -- the true total output data leaving the crossbars. 'col_noc/moved' shows")
print("         col_noc captures ~1.0 of it on wide/square grids but only a fraction on tall")
print("         grids (ffn2 ~0.25, av ~0.5): the mapper splits the C-reduction across levels,")
print("         so col_noc alone is a shape-dependent slice, NOT the total. Use output_moved.")
val_df


VALIDATION: equations vs simulator-measured NoC volumes
               eq_input_B  sim_input_B  in_ratio  output_moved_B  sim_col_noc_B  col_noc/moved
layer                                                                                         
qkv_proj            98304        98250     0.999         7077888        7074000           1.00
qk                  98304        98250     0.999          393216         393000           1.00
av                 196608       196250     0.998          393216         196500           0.50
attn_out_proj       98304        98250     0.999         2359296        2358000           1.00
ffn1                98304        98250     0.999         9437184        9432000           1.00
ffn2               393216       393750     1.001         9437184        2358000           0.25

Input  : sim/eq = 1.0  ->  input equation VALIDATED against the simulator.
Output : output_moved = tiles x (array_col x out_bits/8) x P = output_size x C/array_row
         -- the tru

,eq_input_B,sim_input_B,in_ratio,output_moved_B,sim_col_noc_B,col_noc/moved
layer,,,,,,
qkv_proj,98304,98250,0.999,7077888,7074000,1.00
qk,98304,98250,0.999,393216,393000,1.00
av,196608,196250,0.998,393216,196500,0.50
attn_out_proj,98304,98250,0.999,2359296,2358000,1.00
ffn1,98304,98250,0.999,9437184,9432000,1.00
ffn2,393216,393750,1.001,9437184,2358000,0.25


## Export the I/O tables

`chip_io` and `crossbar_level_io` are per encoder-layer (one pass through the six unique
matmul shapes). All 12 encoder layers share identical geometry, so multiply by `N_LAYERS`
for full-model totals.

In [7]:
import os

OUT_DIR = os.path.join('..', 'outputs', 'bert100m_crossbar_logging')
os.makedirs(OUT_DIR, exist_ok=True)

chip_io.to_csv(os.path.join(OUT_DIR, 'chip_level_io.csv'))
xbar_io.to_csv(os.path.join(OUT_DIR, 'crossbar_level_io.csv'))

print(f"Wrote chip_level_io.csv and crossbar_level_io.csv to {os.path.abspath(OUT_DIR)}")


Wrote chip_level_io.csv and crossbar_level_io.csv to /home/workspace/outputs/bert100m_crossbar_logging
